# VinaScreen
For screening multiple ligands

Requirements: 
1. Ligands: sdf, smi or pdb files
2. Receptor: PDB files of receptors in receptor folder
3. Parameters: config.txt of files specifying binding site, exhaustiveness and number of poses per ligand

*Important Note:* There is only one binding site coordinates that are recognized in the config.txt file. For screening multiple receptors with different binding site coordinates, you need to run this one receptor at a time.


## Prepare Receptor and Ligands with Meeko

**Important**

Make sure your receptor has been cleaned of its co-crystallized ligand and water molecules. This script can process cofactors in the receptor. Please double check on your research design if the cofactor is required to be present on your receptor.

### Receptor Preparation

In [18]:
#Input files and define output filenames
inputpdb = 'humCYP51.pdb'    #input receptor file here with .pdb file name extension
output = 'humCYP51'          #type your desired output filename without extension

In [19]:
#fix missing sidechains and alernate structures
!pdbfixer {inputpdb} --output={output}_fix.pdb --add-atoms=heavy --keep-heterogens=all

In [20]:
#convert pdb to pdbqt with meeko
!mk_prepare_receptor.py -i {output}_fix.pdb -o {output}_heme -p

@> 3633 atoms and 1 coordinate set(s) were parsed in 0.02s.
/home/nikka/miniconda3/envs/vina/lib/python3.11/site-packages/meeko/polymer.py:856: RuntimeWarning: Input residues {'A:601': 'HEM'} not in residue_templates

  warnings.warn(err, RuntimeWarning)
/home/nikka/miniconda3/envs/vina/lib/python3.11/site-packages/meeko/polymer.py:857: RuntimeWarning: Trying to resolve unknown residues by building chemical templates... 
  warnings.warn("Trying to resolve unknown residues by building chemical templates... ", RuntimeWarning)
Molecule contains metal with unspecified charge state -> charging nonmetal coordinated atoms...
All metals will be neutralized and the nonmetal coordinated atoms will be charged according to their explicit valence. 
Total charge of the molecule after recharging: 2

Files written:
humCYP51_heme.pdbqt <-- static (i.e., rigid) receptor input file


In [10]:
#check the presence of heme group in meeko
check='humCYP51_heme.pdbqt'
with open(check, "r") as f:
    for line in f:
        if "HEM" in line and "FE" in line:
            print(line.strip())

ATOM   4418  FE  HEM A 601      19.144  -3.550  22.456  1.00  0.00     1.241 Fe


### For docking protocol validation, prepare control ligand here

In [14]:
# Convert pdb to sdf and add explicit H - single molecule, for the co-crystallized control
from rdkit import Chem

# Define user-input filenames here
input_file = "hum_ctrl_ECL.pdb"
output_file = "ctrlHum_ECL.sdf"

# Read the PDB file using the variable
mol = Chem.MolFromPDBFile(input_file, removeHs=False)

if mol is not None:
    # Add explicit hydrogens AND generate 3D coordinates for them
    mol_with_Hs = Chem.AddHs(mol, addCoords=True)
    
    # Write the molecule to an SDF file using the variable
    with Chem.SDWriter(output_file) as writer:
        writer.write(mol_with_Hs)
    print(f"Successfully added explicit hydrogens and converted to {output_file}!")
else:
    print(f"Error: RDKit could not read the PDB file '{input_file}'.")

Successfully added explicit hydrogens and converted to ctrlHum_ECL.sdf!


In [17]:
!mk_prepare_ligand.py -i {output_file}

/home/nikka/miniconda3/envs/vina/lib/python3.11/site-packages/meeko/molsetup.py:1584: RuntimeWarning: RDKit molecule not labeled as 3D. This warning won't show again.
  warnings.warn(
RDKit molecule has 0 fragments. Must have 1.
Input molecules processed: 0, skipped: 0
PDBQT files written: 0
PDBQT files not written due to error: 1
Input molecules with errors: 1
No PDBQT files were written due to errors!


### Preparation of Ligand for docking

This script accepts one or more ligands in sdf format

In [26]:
#convert sdf library to individual pdbqt files

import os
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation
from dimorphite_dl import protonate_smiles  # Corrected Import

# 1. Define your file paths
input_sdf = "MyristicaG3.sdf"       
output_dir = "ligand"     
target_ph = 7.0  #CNS pH during PAM

os.makedirs(output_dir, exist_ok=True)
supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)
mk_prep = MoleculePreparation()

for mol in supplier:
    if mol is None:
        continue
        
    mol_name = mol.GetProp('_Name').strip() if (mol.HasProp('_Name') and mol.GetProp('_Name').strip()) else f"ligand_{mol.GetNumAtoms()}_atoms"

    try:
        # Convert RDKit mol to SMILES for Dimorphite-DL
        original_smiles = Chem.MolToSmiles(mol)
        
        # 2. Fix Protonation State at Physiological pH
        # Returns a list of protonated SMILES strings
        protonated_smiles_list = protonate_smiles(
            original_smiles, 
            ph_min=target_ph - 0.5, 
            ph_max=target_ph + 0.5
        )
        
        if not protonated_smiles_list:
            print(f"Skipping {mol_name}: Could not determine protonation state.")
            continue
            
        # Take the first predicted protonation state
        best_smiles = protonated_smiles_list[0]
        
        # Convert back to RDKit molecule
        best_mol = Chem.MolFromSmiles(best_smiles)
        
        # 3. Add explicit hydrogens corresponding to the new protonation state
        best_mol = Chem.AddHs(best_mol)
        
        # 4. Generate a new 3D conformation
        embed_status = AllChem.EmbedMolecule(best_mol, AllChem.ETKDGv3())
        if embed_status != 0:
             print(f"Warning: Conformer generation failed for {mol_name}. Proceeding with fallback.")
             AllChem.Compute2DCoords(best_mol) 
        
        AllChem.MMFFOptimizeMolecule(best_mol)

        # 5. Process into PDBQT format
        mk_prep.prepare(best_mol)
        pdbqt_string = mk_prep.write_pdbqt_string()
        
        output_filepath = os.path.join(output_dir, f"{mol_name}.pdbqt")
        with open(output_filepath, "w") as f:
            f.write(pdbqt_string)
            
        print(f"Successfully created: {output_filepath}")
        
    except Exception as e:
        print(f"Error processing {mol_name}: {e}")

/home/nikka/miniconda3/envs/vina/lib/python3.11/site-packages/meeko/preparation.py:693: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWriterLegacy.write_string(). MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
/home/nikka/miniconda3/envs/vina/lib/python3.11/site-packages/meeko/preparation.py:467: DeprecationWarning: MoleculePreparation.setup is deprecated in Meeko v0.5. MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)


Successfully created: ligand/g3771a036a224a462.pdbqt
Successfully created: ligand/g3771a036a224a545.pdbqt
Successfully created: ligand/g3771a036a224a224.pdbqt
Successfully created: ligand/g3771a036a224a399.pdbqt
Successfully created: ligand/g3771a036a224a231.pdbqt
Successfully created: ligand/g3771a036a224a242.pdbqt
Successfully created: ligand/g3771a036a224a094.pdbqt
Successfully created: ligand/g3771a036a224a148.pdbqt
Successfully created: ligand/g3771a036a224a237.pdbqt
Successfully created: ligand/g3771a036a224a030.pdbqt
Successfully created: ligand/g3771a036a224a440.pdbqt
Successfully created: ligand/g3771a036a224a311.pdbqt
Successfully created: ligand/g3771a036a224a286.pdbqt
Successfully created: ligand/g3771a036a224a404.pdbqt
Successfully created: ligand/g3771a036a224a033.pdbqt
Successfully created: ligand/g3771a036a224a446.pdbqt
Successfully created: ligand/g3771a036a030a462.pdbqt
Successfully created: ligand/g3771a036a030a545.pdbqt
Successfully created: ligand/g3771a036a030a121

/home/nikka/miniconda3/envs/vina/lib/python3.11/site-packages/meeko/molsetup.py:1584: RuntimeWarning: RDKit molecule not labeled as 3D. This warning won't show again.
  warnings.warn(


Successfully created: ligand/g3771a036a224b151.pdbqt
Successfully created: ligand/g3771a036a224b433.pdbqt
Successfully created: ligand/g3771a036a224b044.pdbqt
Successfully created: ligand/g3771a036a224b289.pdbqt
Successfully created: ligand/g3771a036a224b514.pdbqt
Successfully created: ligand/g3771a036a224b116.pdbqt
Successfully created: ligand/g3771a036a224b269.pdbqt
Successfully created: ligand/g3771a036a224b185.pdbqt
Successfully created: ligand/g3771a036a224b298.pdbqt
Successfully created: ligand/g3771a036a224b545.pdbqt
Successfully created: ligand/g3771a036a224b462.pdbqt
Successfully created: ligand/g3771a036a224b100.pdbqt


## Convert ligands to pdbqt (inhouse script - obsolete)
!! output will be in ligand folder. make sure it is empty or previous run renamed

In [1]:
!python pdbqt_converter.py /home/nikka/Frag_output/top10_Ligs/top10Ligs.sdf

Detecting file format: sdf
Reading molecules from: /home/nikka/Frag_output/top10_Ligs/top10Ligs.sdf
Found 41 molecules
Output directory: ligand
Processed: 41/41 (converted: 41, skipped: 0) Rate: 72.7 mol/s ETA: 0s
CONVERSION SUMMARY
Total molecules processed: 41
Successfully converted: 41
Skipped (errors/timeouts): 0
Success rate: 100.0%
Total time: 0.6 seconds
Output files saved in: ligand/


## Run docking screen
do not forget to edit config.txt!!!!!
rename output

In [27]:
!python run_vinascreen.py

VinaScreen High-Throughput Docking Pipeline (main.py)

Step 1: Validating environment setup...
Validating VinaScreen environment...
✓ Found Vina executable: vina_1.2.7_linux_x86_64
✓ Found ligand folder with 78 ligand files
✓ Found receptor folder with 2 receptor files
✓ Found and validated config.txt file
✓ Created output folder: output
✓ Environment validation successful!

Step 2: Loading inputs...
  78 ligands loaded
  2 receptors loaded

Step 3: Initializing CSV reporter and progress tracker...
  Output CSV: vinascreen_results.csv
  Total docking jobs: 156

Step 4: Launching docking runs...

Processed: 156/156 (kept: 156, skipped: 0)

Step 5: Summary
Docking complete. 156 jobs processed.
Results saved to: vinascreen_results.csv


## Evaluation of Docking Control

In [25]:
# Validation of control
from rdkit import Chem
from rdkit.Chem import rdMolAlign, AllChem
import subprocess, os

# ==========================================
#               USER SETTINGS
# ==========================================
# 1. The PDBQT file you got from docking
docked_pdbqt_file = "output/humCYP51_heme_ctrlHum_ECN.pdbqt"

# 2. The crystal structure (reference) you want to compare against
crystal_reference_pdb = "hum_ctrlECN_complex.pdb"

# 3. Where to save the temporary converted SDF file
temp_sdf_file = "hum_ctrl_docked_poses.sdf"

# 4. Where to save the final text report
text_report_file = "hum_validation_report.txt"

# 5. RMSD cutoff for a successful pose (in Angstroms)
rmsd_cutoff = 2.0
# ==========================================


# ── Step 1: Convert the docked PDBQT output to SDF using Meeko ──
subprocess.run([
    "mk_export.py",
    docked_pdbqt_file,
    "-s", temp_sdf_file
], check=True)

# ── Step 2: Load the crystal reference ──
ref = Chem.MolFromPDBFile(crystal_reference_pdb, removeHs=True, sanitize=True)
if ref is None:
    # fallback: try reading as SDF if you converted it earlier
    # Attempting to load using the same basename as the PDB file
    fallback_sdf = crystal_reference_pdb.replace('.pdb', '.sdf')
    ref = Chem.SDMolSupplier(fallback_sdf, removeHs=True)[0]

# ── Step 3: Calculate RMSD for each docked pose ──
suppl = Chem.SDMolSupplier(temp_sdf_file, removeHs=True)

# Prepare the report text
report_lines = []
report_lines.append("")
report_lines.append("=" * 55)
report_lines.append("  DOCKING PROTOCOL VALIDATION REPORT")
report_lines.append("=" * 55)
report_lines.append(f"  {'Pose':<8} {'RMSD (Å)':<14} {'Result'}")
report_lines.append("-" * 55)

best_rmsd = None
best_pose = None
pass_count = 0

for i, pose in enumerate(suppl):
    if pose is None:
        report_lines.append(f"  Pose {i+1:<4} {'SKIPPED':<14} Could not parse pose")
        continue
    try:
        # rdMolAlign.CalcRMS uses MCS-based atom mapping — handles
        # atom order differences between PDB and PDBQT automatically
        rmsd = rdMolAlign.CalcRMS(pose, ref)
        status = "PASS  <==" if rmsd <= rmsd_cutoff else "FAIL"
        
        if rmsd <= rmsd_cutoff:
            pass_count += 1
        if best_rmsd is None or rmsd < best_rmsd:
            best_rmsd = rmsd
            best_pose = i + 1
            
        report_lines.append(f"  Pose {i+1:<4} {rmsd:<14.3f} {status}")
    except Exception as e:
        report_lines.append(f"  Pose {i+1:<4} {'ERROR':<14} {str(e)}")

report_lines.append("=" * 55)
report_lines.append(f"  Best pose  : Pose {best_pose} ({best_rmsd:.3f} Å)" if best_pose else "  Best pose  : None")
report_lines.append(f"  Poses pass : {pass_count} / {i+1}")
report_lines.append(f"  Verdict    : {'PROTOCOL VALIDATED' if pass_count > 0 else 'PROTOCOL FAILED'}")
report_lines.append("=" * 55)

# Join the lines into a single string
full_report = "\n".join(report_lines)

# Print to the console
print(full_report)

# Write to the output text file
with open(text_report_file, "w") as f:
    f.write(full_report)

print(f"\nReport successfully saved to {text_report_file}")


  DOCKING PROTOCOL VALIDATION REPORT
  Pose     RMSD (Å)       Result
-------------------------------------------------------
  Pose 1    2.452          FAIL
  Pose 2    4.008          FAIL
  Pose 3    1.918          PASS  <==
  Pose 4    4.611          FAIL
  Pose 5    2.669          FAIL
  Pose 6    4.178          FAIL
  Pose 7    1.704          PASS  <==
  Pose 8    2.855          FAIL
  Pose 9    4.494          FAIL
  Pose 10   6.259          FAIL
  Pose 11   4.101          FAIL
  Pose 12   6.584          FAIL
  Pose 13   7.675          FAIL
  Pose 14   5.364          FAIL
  Pose 15   6.186          FAIL
  Best pose  : Pose 7 (1.704 Å)
  Poses pass : 2 / 15
  Verdict    : PROTOCOL VALIDATED

Report successfully saved to hum_validation_report.txt


In [24]:
#Validation of control

from rdkit import Chem
from rdkit.Chem import rdMolAlign, AllChem
import subprocess, os

# ── Step 1: Convert the docked PDBQT output to SDF using Meeko ──
subprocess.run([
    "mk_export.py",
    "output/NfCYP51_heme_ctrl.pdbqt",
    "-s", "docked_poses.sdf"
], check=True)

# ── Step 2: Load the crystal reference ──
ref = Chem.MolFromPDBFile("ctrl_5tv.pdb", removeHs=True, sanitize=True)
if ref is None:
    # fallback: try reading as SDF if you converted it earlier
    ref = Chem.SDMolSupplier("ctrl_5tv.sdf", removeHs=True)[0]

# ── Step 3: Calculate RMSD for each docked pose ──
suppl = Chem.SDMolSupplier("docked_poses.sdf", removeHs=True)

print("")
print("=" * 55)
print("  DOCKING PROTOCOL VALIDATION REPORT")
print("=" * 55)
print(f"  {'Pose':<8} {'RMSD (Å)':<14} {'Result'}")
print("-" * 55)

cutoff    = 2.0
best_rmsd = None
best_pose = None
pass_count = 0

for i, pose in enumerate(suppl):
    if pose is None:
        print(f"  Pose {i+1:<4} {'SKIPPED':<14} Could not parse pose")
        continue
    try:
        # rdMolAlign.CalcRMS uses MCS-based atom mapping — handles
        # atom order differences between PDB and PDBQT automatically
        rmsd = rdMolAlign.CalcRMS(pose, ref)
        status = "PASS  <==" if rmsd <= cutoff else "FAIL"
        if rmsd <= cutoff:
            pass_count += 1
        if best_rmsd is None or rmsd < best_rmsd:
            best_rmsd = rmsd
            best_pose = i + 1
        print(f"  Pose {i+1:<4} {rmsd:<14.3f} {status}")
    except Exception as e:
        print(f"  Pose {i+1:<4} {'ERROR':<14} {str(e)}")

print("=" * 55)
print(f"  Best pose  : Pose {best_pose} ({best_rmsd:.3f} Å)")
print(f"  Poses pass : {pass_count} / {i+1}")
print(f"  Verdict    : {'PROTOCOL VALIDATED' if pass_count > 0 else 'PROTOCOL FAILED'}")
print("=" * 55)


  DOCKING PROTOCOL VALIDATION REPORT
  Pose     RMSD (Å)       Result
-------------------------------------------------------
  Pose 1    0.943          PASS  <==
  Pose 2    0.948          PASS  <==
  Pose 3    6.926          FAIL
  Pose 4    4.792          FAIL
  Pose 5    6.835          FAIL
  Pose 6    6.433          FAIL
  Pose 7    6.516          FAIL
  Pose 8    7.043          FAIL
  Pose 9    2.461          FAIL
  Pose 10   6.422          FAIL
  Best pose  : Pose 1 (0.943 Å)
  Poses pass : 2 / 10
  Verdict    : PROTOCOL VALIDATED


# Convert VinaScreen pdbqt output to smiles

In [2]:
import os
import re
import pandas as pd
from openbabel import pybel

def parse_vina_pdbqt(filename):
    """Extract models and scores from vina PDBQT output file."""
    results = []
    with open(filename, 'r') as f:
        content = f.read()
    # Split models: each model starts with "MODEL" and ends with "ENDMDL"
    models = re.split('(MODEL.*?ENDMDL)', content, flags=re.DOTALL)
    model_number = 1
    for model in models:
        if 'MODEL' in model and 'ENDMDL' in model:
            # Extract vina score (affinity) from REMARK line
            score_match = re.search(r'REMARK VINA RESULT:\s*([-.\d]+)', model)
            if score_match:
                score = float(score_match.group(1))
            else:
                score = None
            # Write each pose to temp pdbqt for conversion
            temp_name = '_temp_model.pdbqt'
            with open(temp_name, 'w') as out:
                out.write(model)
            # Use openbabel to convert to SMILES
            mols = list(pybel.readfile("pdbqt", temp_name))
            if mols:
                smiles = mols[0].write(format="smi").split()[0]
            else:
                smiles = None
            os.remove(temp_name)
            results.append((model_number, score, smiles))
            model_number += 1
    return results

data = []
folder = "output_PDL1G1" #path to folder
for fname in os.listdir(folder):
    if fname.endswith(".pdbqt"):
        full_path = os.path.join(folder, fname)
        basename = os.path.splitext(fname)[0]
        models = parse_vina_pdbqt(full_path)
        for model_id, vina_score, smiles in models:
            row_id = f"{basename}_{model_id}"
            data.append({'ID': row_id, 'SMILES': smiles, 'vina_score': vina_score})

# Output as CSV
df = pd.DataFrame(data)

# Sort to ensure lowest model number comes first for each SMILES
df['model_num'] = df['ID'].str.extract(r'_(\d+)$').astype(int)
df.sort_values(by=['SMILES', 'model_num'], inplace=True)
# Drop duplicates keeping lowest model number (first occurrence)
df_no_dupes = df.drop_duplicates(subset=["SMILES"], keep='first')
# Drop auxiliary column before saving
df_no_dupes = df_no_dupes.drop(columns=['model_num'])
df_no_dupes.to_csv("vina_PDL1G1.csv", index=False) #change file name


*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is pdL1G1_PdL1G1_0250)

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is pdL1G1_PdL1G1_0250)

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is pdL1G1_PdL1G1_0250)

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is pdL1G1_PdL1G1_0320)

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is pdL1G1_PdL1G1_0320)

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is pdL1G1_PdL1G1_0320)

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is pdL1G1_PdL1G1_0202)

*** Op